# 🧪 OCR Evaluation Masterclass (Module B)
**Table 2.3.2 — OCR Model Variable Test**

This notebook evaluates three OCR configurations for extracting handwritten student answers. The results will be logged to `outputs/ocr_metrics/eval_metrics_scaffold.json`.

### Configurations Tested:
- **Config A**: Tesseract (LSTM Handwriting Mode)
- **Config B**: TrOCR (Base / Small)
- **Config C**: PaddleOCR (Handwriting Mode)

### Pre-requisites:
Ensure you have run `python download_ocr_models.py` successfully (even if on a non-ML machine, ensure CPU packages are installed via `requirements_ocr.txt`).

In [ ]:
import os
import sys
import cv2
from PIL import Image
import matplotlib.pyplot as plt

# Add src to path
sys.path.append(os.path.abspath('./src'))

from image_preprocessor import preprocess_for_ocr
from ocr_engines import TesseractEngine, TrOCREngine, PaddleEngine
from metrics import calculate_wer, calculate_cer, is_dropped

## 1. Load and Pre-process Sample Image
We apply CLAHE, Deskewing, and Otsu Binarization.

In [ ]:
# Default OCR eval data location (repo-wide): Unified_Datasets/OCR/test_images/
# This keeps Module B aligned with the unified dataset convention.
from pathlib import Path

REPO_ROOT = Path(os.getcwd()).resolve().parents[1]  # .../SwiftGrade-Models
UNIFIED_OCR_DIR = REPO_ROOT / 'Unified_Datasets' / 'OCR'
TEST_IMAGES_DIR = UNIFIED_OCR_DIR / 'test_images'
TEST_IMAGES_DIR.mkdir(parents=True, exist_ok=True)

test_image_path = str(TEST_IMAGES_DIR / 'sample_handwriting.png')

# Create a dummy test image for demonstration if one doesn't exist
if not os.path.exists(test_image_path):
    import numpy as np
    # Create a blank white image
    img = np.ones((200, 800, 3), dtype=np.uint8) * 255
    # Add some text (as a proxy for handwriting in this dummy generation)
    cv2.putText(
        img,
        'The mitochondria is the powerhouse of the cell.',
        (50, 100),
        cv2.FONT_HERSHEY_SCRIPT_SIMPLEX,
        1,
        (0, 0, 0),
        2,
    )
    cv2.imwrite(test_image_path, img)
    print(f"Created dummy test image at {test_image_path}")

# Run Pre-processing
original_img, binary_img = preprocess_for_ocr(test_image_path)

# Display results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
ax1.imshow(cv2.cvtColor(original_img, cv2.COLOR_BGR2RGB))
ax1.set_title('Original (Deskewed)')
ax1.axis('off')

ax2.imshow(binary_img, cmap='gray')
ax2.set_title('Binarized (Otsu)')
ax2.axis('off')
plt.show()

## 2. Initialize OCR Engines
*(Note: TrOCR and PaddleOCR might take a moment to load weights into memory)*

In [ ]:
print("Initializing Engines...")
engine_tess = TesseractEngine()
# engine_trocr = TrOCREngine(model_size="small") # Uncomment to load TrOCR
# engine_paddle = PaddleEngine()                 # Uncomment to load PaddleOCR
print("Done.")

## 3. Run Inference & Calculate Metrics
Compare outputs against the Ground Truth.

In [ ]:
ground_truth = "The mitochondria is the powerhouse of the cell."

# Convert CV2 image to PIL for engines that need it
pil_original = Image.fromarray(cv2.cvtColor(original_img, cv2.COLOR_BGR2RGB))
pil_binary = Image.fromarray(binary_img)

results = {}

# --- Config A: Tesseract ---
if engine_tess.pytesseract:
    # Tesseract usually performs better on binarized text for scanned docs
    pred_tess = engine_tess.run(pil_binary)
    results["A_tesseract"] = pred_tess

# --- Config B: TrOCR ---
# if engine_trocr.model_loaded:
#     # TrOCR prefers RGB un-binarized images
#     pred_trocr = engine_trocr.run(pil_original)
#     results["B_trocr"] = pred_trocr

# --- Config C: PaddleOCR ---
# if engine_paddle.model_loaded:
#     # PaddleOCR works with CV2 numpy arrays
#     pred_paddle = engine_paddle.run(original_img)
#     results["C_paddleocr"] = pred_paddle

for model, pred in results.items():
    wer = calculate_wer(ground_truth, pred)
    cer = calculate_cer(ground_truth, pred)
    dropped = is_dropped(pred)
    
    print(f"\n{'='*40}")
    print(f"Model: {model}")
    print(f"Prediction: '{pred}'")
    print(f"WER: {wer:.4f} | CER: {cer:.4f} | Dropped: {dropped}")

## 4. Batch Evaluation (To be implemented)
In a full run, you would iterate over `OCR/test_images/`, aggregate the WER/CER scores, and calculate the overall Drop Rate. Then dump the results to `OCR/outputs/ocr_metrics/eval_metrics_scaffold.json`.